# Watem NB03 — Compare and evaluate
Compare independent scenarios for the same site, annual erosion and identical 20/40/60 mm storm shapes. Results are pilot estimates; automatic outlet is not guaranteed a measured physical outlet.

In [ ]:
from google.colab import drive,files
from pathlib import Path
import os, shutil, json, re, zipfile, tempfile
import numpy as np, pandas as pd, geopandas as gpd, rasterio, matplotlib.pyplot as plt
from rasterio.features import rasterize
from shapely.ops import unary_union
from shapely.geometry import box
from IPython.display import display
drive.mount('/content/drive')
SITE_ID='my_catchment' #@param {type:'string'}
assert SITE_ID and SITE_ID.replace('_','').replace('-','').isalnum()
ROOT=Path('/content/drive/MyDrive/Watem')
SITE=ROOT/'student_sites'/SITE_ID; BASELINE=SITE/'baseline'
assert (BASELINE/'complete.json').exists(),'Complete NB01 first'
import urllib.request, sys
helper_dir=Path('/content/watem_student_helpers'); helper_dir.mkdir(exist_ok=True)
urllib.request.urlretrieve('https://raw.githubusercontent.com/kvanoost/Watem/683decfd6ad9bced8cd0b431a772b167d583291e/watem_student_common.py',helper_dir/'watem_student_common.py')
sys.path.insert(0,str(helper_dir))
import watem_student_common as ws


In [ ]:
# Discover completed scenarios only; same site and rainfall shape.
folders={'baseline':BASELINE}
folders.update({p.name:p for p in (SITE/'scenarios').glob('*') if (p/'complete.json').exists()})
print('Available:',list(folders))
SELECT='all' #@param {type:'string'}
names=list(folders) if SELECT=='all' else ['baseline']+[x.strip() for x in SELECT.split(',') if x.strip()!='baseline']
assert all(x in folders for x in names)
frames=[]
for name in names:
    df=pd.read_csv(folders[name]/'summary.csv'); df.insert(0,'scenario',name); frames.append(df)
df=pd.concat(frames,ignore_index=True)
for key in ['erosion_t','river_t','export_t','runoff_m3','peak_m3_s']:
    if key not in df: continue
    ref=df[df.scenario=='baseline'].set_index('mode')[key]
    df[key+'_reduction_pct']=100*(1-df[key]/df['mode'].map(ref).replace(0,np.nan))
display(df.round(3))
COMP=SITE/'comparison'; COMP.mkdir(exist_ok=True); df.to_csv(COMP/'scenario_summary.csv',index=False)
for mode,g in df.groupby('mode'):
    metrics=['erosion_t','river_t','export_t']+(['runoff_m3','peak_m3_s'] if mode.startswith('event') else [])
    fig,ax=plt.subplots(1,len(metrics),figsize=(4*len(metrics),4),squeeze=False,constrained_layout=True)
    for a,k in zip(ax.flat,metrics): g.set_index('scenario')[k].plot.bar(ax=a,rot=30,title=k)
    fig.savefig(COMP/f'{mode}_metrics.png',dpi=160); plt.show()
    if mode.startswith('event'):
        fig,ax=plt.subplots(figsize=(10,4))
        for name in names:
            h=pd.read_csv(folders[name]/mode/'report/hydrograph.csv'); ax.plot(h.time_min,h.discharge_m3_s,label=name)
        ax.set(xlabel='Time (min)',ylabel='Discharge (m³/s)',title=mode+' | automatic model outlet'); ax.legend(); fig.savefig(COMP/f'{mode}_hydrographs.png',dpi=160); plt.show()
    for raster in ['erosion_t_ha']+(['local_runoff_mm'] if mode.startswith('event') else []):
        arrays=[ws.read_raster(folders[name]/mode/'report'/f'{raster}.tif')[0] for name in names]
        vals=np.concatenate([a[a!=-9999] for a in arrays]); lim=max(float(np.percentile(np.abs(vals),98)),.01)
        fig,axes=plt.subplots(1,len(names),figsize=(5*len(names),5),squeeze=False,constrained_layout=True)
        for a,name,v in zip(axes.flat,names,arrays):
            im=a.imshow(np.ma.masked_equal(v,-9999),cmap='RdBu' if raster=='erosion_t_ha' else 'Blues',vmin=-lim if raster=='erosion_t_ha' else 0,vmax=lim); a.set_title(name); a.axis('off')
        fig.colorbar(im,ax=list(axes.flat),shrink=.6); fig.savefig(COMP/f'{mode}_{raster}.png',dpi=160); plt.show()
files.download(shutil.make_archive(str(SITE/'comparison'),'zip',COMP))
